# Add Security Group as Viewer to All Workspaces

Run this notebook in Fabric after workspaces have been created. It reads the workspace IDs from `teams-resolved.xlsx` and adds a specified security group as **Viewer** to all team workspaces.

## Configuration

Edit the values below:
- `RESULTS_FILE`: Path to the workspaces Excel file generated by the creation script
- `SECURITY_GROUP_ID`: Entra security group ID to add as viewer
- `SECURITY_GROUP_NAME`: Display name of the security group (for logging)
- `DRY_RUN`: Set to False to actually make changes

In [ ]:
# Configuration — edit these
RESULTS_FILE = "Files/teams-resolved.xlsx"  # Output from 01-create-workspaces.py
SECURITY_GROUP_ID = "12345678-1234-1234-1234-123456789012"  # Entra group ID
SECURITY_GROUP_NAME = "All-Players-Viewer"  # Display name for logging
DRY_RUN = True  # Set to False to actually make changes

## Step 1: Load Workspace Data

In [ ]:
import pandas as pd
import notebookutils

# Load the results file
df = pd.read_excel(f"/lakehouse/default/{RESULTS_FILE}")
required_columns = {"TeamName", "WorkspaceId"}
missing = required_columns.difference(df.columns)
if missing:
    raise ValueError(
        f"The workbook must include WorkspaceName and WorkspaceId columns. Missing: {missing}"
    )

# Get unique workspaces
workspaces = df.drop_duplicates(subset=["TeamName"])[["TeamName", "WorkspaceId"]].to_dict(orient="records")
print(f"Found {len(workspaces)} workspaces:")
for ws in workspaces:
    print(f"  - {ws['TeamName']}: {ws['WorkspaceId']}")

## Step 2: Get Fabric API Token

In [ ]:
# Get token for Fabric REST API
fabric_token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
fabric_headers = {
    "Authorization": f"Bearer {fabric_token}",
    "Content-Type": "application/json",
}
print("✅ Token acquired")

## Step 3: Add Security Group as Viewer to All Workspaces

In [ ]:
import requests
import json

print(f"Adding {SECURITY_GROUP_NAME} ({SECURITY_GROUP_ID}) as Viewer to all workspaces...\n")

if DRY_RUN:
    print("🔍 DRY RUN MODE - No changes will be made\n")

results = []

for ws in workspaces:
    team_name = ws["TeamName"]
    workspace_id = ws["WorkspaceId"]
    
    if DRY_RUN:
        print(f"[DRY RUN] Would add {SECURITY_GROUP_NAME} as Viewer to {team_name}")
        results.append({
            "TeamName": team_name,
            "WorkspaceId": workspace_id,
            "GroupId": SECURITY_GROUP_ID,
            "GroupName": SECURITY_GROUP_NAME,
            "Role": "Viewer",
            "Status": "DRY_RUN"
        })
    else:
        # Add group as Viewer
        payload = {
            "principal": {
                "id": SECURITY_GROUP_ID,
                "type": "Group",
            },
            "role": "Viewer",  # Viewers for security group (e.g., observers)
        }

        response = requests.post(
            f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/roleAssignments",
            headers=fabric_headers,
            json=payload,
        )

        if response.status_code == 200:
            print(f"✅ {team_name}: Added {SECURITY_GROUP_NAME} as Viewer")
            status = "SUCCESS"
        else:
            print(f"❌ {team_name}: Failed ({response.status_code})")
            print(f"   {response.text}")
            status = "FAILED"

        results.append({
            "TeamName": team_name,
            "WorkspaceId": workspace_id,
            "GroupId": SECURITY_GROUP_ID,
            "GroupName": SECURITY_GROUP_NAME,
            "Role": "Viewer",
            "Status": status
        })

print(f"\n✨ Complete! {len([r for r in results if r['Status'] == 'SUCCESS'])} workspaces updated.")

## Summary

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(results)
display(summary_df)

success_count = len([r for r in results if r["Status"] == "SUCCESS"])
failed_count = len([r for r in results if r["Status"] == "FAILED"])
print(f"\nSummary: {success_count} success, {failed_count} failed, {len([r for r in results if r['Status'] == 'DRY_RUN'])} dry-run")